In [11]:
# 끊어짐 이슈로 드라이브 마운팅 필수
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# 코랩환경 기준

# 데이터 파싱을 위한 패키지 설치
!pip install wikiextractor gensim tqdm -q

# python-mecab-ko : 한국어 사전 포함 올인원 패키지
!pip install python-mecab-ko -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.6/579.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 24.2 MB/s eta 0:00:00


In [13]:
!find /usr -name "dicrc" 2>/dev/null


/usr/local/lib/python3.12/dist-packages/mecab_ko_dic/dictionary/dicrc


In [14]:
# 형태소 분석기 동작 확인
from mecab import MeCab
mecab = MeCab()
print(mecab.morphs('대한민국의 수도는 서울입니다'))


['대한민국', '의', '수도', '는', '서울', '입니다']


In [15]:
# wikiextractor 정규식 호환성 패치 (Python 3.12 대응)
from pathlib import Path

p = Path('/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py')
backup = p.with_suffix('.py.bak')

# 백업
if not backup.exists():
    backup.write_text(p.read_text(encoding='utf-8'), encoding='utf-8')

lines = p.read_text(encoding='utf-8').splitlines(keepends=True)

new_lines = []
i = 0
patched_1 = False
patched_2 = False

while i < len(lines):
    s = lines[i].lstrip()

    if s.startswith('ExtLinkBracketedRegex = re.compile('):
        indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
        new_lines.extend([
            indent + 'ExtLinkBracketedRegex = re.compile(\n',
            indent + "    '\\[((' + '|'.join(wgUrlProtocols) + ')' + EXT_LINK_URL_CLASS + r'+)\\s*([^\\]\\x00-\\x08\\x0a-\\x1F]*?)\\]',\n",
            indent + '    re.S | re.U | re.I)\n',
        ])
        i += 3
        patched_1 = True
        continue

    if s.startswith('EXT_IMAGE_REGEX = re.compile('):
        indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
        new_lines.extend([
            indent + 'EXT_IMAGE_REGEX = re.compile(\n',
            indent + '    r"""^(http://|https://)([^][<>"\\x00-\\x20\\x7F\\s]+)\n',
            indent + '    /([A-Za-z0-9_.,~%\\-+&;#*?!=()@\\x80-\\xFF]+)\\.(gif|png|jpg|jpeg)$""",\n',
            indent + '    re.X | re.S | re.U | re.I)\n',
        ])
        i += 4
        patched_2 = True
        continue

    new_lines.append(lines[i])
    i += 1

p.write_text(''.join(new_lines), encoding='utf-8')

print('backup:', backup)
print('patched ExtLinkBracketedRegex:', patched_1)
print('patched EXT_IMAGE_REGEX:', patched_2)
print('done')

backup: /usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py.bak
patched ExtLinkBracketedRegex: True
patched EXT_IMAGE_REGEX: True
done


In [16]:
%cd /content

# 위키피디아 덤프(위키피디아 데이터) 다운로드
!wget -q --show-progress https://dumps.wikimedia.org/kowiki/latest/kowiki-latest-pages-articles.xml.bz2

/content
kowiki-latest-pages 100%[===================>]   1.17G  3.22MB/s    in 7m 16s  


In [17]:
# 위키익스트랙터 를 이용한 위키피디아 덤프 파싱
!python -m wikiextractor.WikiExtractor /content/kowiki-latest-pages-articles.xml.bz2 -o /content/text --processes 4

/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:33: SyntaxWarning: invalid escape sequence '\w'
  tailRE = re.compile('\w+')
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:171: SyntaxWarning: invalid escape sequence '\.'
  text = re.sub(u' (,:\.\)\]»)', r'\1', text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:172: SyntaxWarning: invalid escape sequence '\['
  text = re.sub(u'(\[\(«) ', r'\1', text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:379: SyntaxWarning: invalid escape sequence '\['
  '\[((' + '|'.join(wgUrlProtocols) + ')' + EXT_LINK_URL_CLASS + r'+)\s*([^\]\x00-\x08\x0a-\x1F]*?)\]',
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:733: SyntaxWarning: invalid escape sequence '\w'
  return re.sub("&#?(\w+);", fixup, text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:1278: SyntaxWarning: invalid escape sequence '\['
  reOpen = re.compile('{{2,}|\[{2,}')
/usr/local/lib/

In [18]:
# 현재 경로에 있는 디렉터리와 파일들 의 리스트 받아오기
%ls
%ls text

drive/  kowiki-latest-pages-articles.xml.bz2  sample_data/  text/
AA/  AB/  AC/  AD/  AE/  AF/  AG/  AH/  AI/  AJ/  AK/  AL/  AM/  AN/


In [19]:
# 운영체제 기능 사용
import os
# 정규표현
import re

In [20]:
# AA- AF 디렉토리 안의 wiki 숫자 형태의 수많은 파일들을 하나로 통합하는 과정 진행
# AA~ AF 디렉토리 안 모든 파일들의 경로를 리스트 형태로 저장

def list_wiki(dirname):
    filepaths = []
    filenames = os.listdir(dirname)

    for filename in filenames:
        filepath = os.path.join(dirname, filename)

        if os.path.isdir(filepath):
            filepaths.extend(list_wiki(filepath))
        else:
            find = re.findall(r'wiki_[0-9][0-9]', filepath)
            if len(find) > 0:
                filepaths.append(filepath)

    return sorted(filepaths)

In [21]:
# 총 파일의 개수 확인
filepaths = list_wiki('text')
len(filepaths)

1354

In [22]:
# output_file.txt 에 850개 파일을 전부 합치기
with open('output_file.txt', 'w') as outfile:
    for filename in filepaths:
        with open(filename) as infile:
            contents = infile.read()
            outfile.write(contents)

In [23]:
# 위키 파싱 + 파일 병합 완료 후 드라이브 저장
import shutil
shutil.copy('output_file.txt', '/content/drive/MyDrive/output_file.txt')
print('Drive 저장 완료')

Drive 저장 완료


In [28]:
# 형태소 분석
from tqdm import tqdm
from mecab import MeCab

# Mecab을 사용한 토큰화 진행
mecab = MeCab()

# out_file 에는 총 몇줄이 있을까
f = open('output_file.txt', encoding='utf-8')
lines = f.read().splitlines()
f.close()
print(len(lines))


13263501


In [30]:
# 아무런 단어도 들어있지 않은 '' 와 같은 줄도 존재한다.
# 제외하고 형태소 분석을 수행한다.

result = []

for line in tqdm(lines):
    # 빈 문자열이 아닌 경우에만 수행
    if line:
        result.append(mecab.morphs(line))


 32%|███▏      | 4198705/13263501 [1:01:40<2:13:10, 1134.52it/s]


KeyboardInterrupt: 

코랩서버 시스템 RAM
12.2 / 12.7 GB.. 중단 후 학습 시작 (1시간 실행. 런타임 끊길 우려 있음. 중단 후 학습)

In [5]:
f = open('output_file.txt', encoding='utf-8')
lines = f.read().splitlines()
f.close()
print(len(lines))


13263501


In [7]:
from tqdm import tqdm
from mecab import MeCab

# Mecab을 사용한 토큰화 진행
mecab = MeCab()

# 20만개로 줄여서 진행 (RAM 절약)
result = []

for line in tqdm(lines[:200000]):
    # 빈 문자열이 아닌 경우에만 수행
    if line:
        result.append(mecab.morphs(line))


100%|██████████| 200000/200000 [03:10<00:00, 1047.31it/s]


In [8]:
# 형태소 분석 완료 후 저장
import pickle
with open('/content/drive/MyDrive/result.pkl', 'wb') as f:
    pickle.dump(result, f)
print('형태소 분석 결과 저장 완료')

형태소 분석 결과 저장 완료


In [9]:
f = open('output_file.txt', encoding='utf-8')

i = 0

while True:
    line = f.readline()
    if line != '\n':
        i = i+1
        print('%d번째 줄 : ' %i + line)
    if i == 10:
        break
# open 을 했다면 꼭 닫아주어야 한다.
f.close()

1번째 줄 : <doc id="5" url="https://ko.wikipedia.org/wiki?curid=5" title="지미 카터">

2번째 줄 : 지미 카터

3번째 줄 : 제임스 얼 "지미 카터 주니어"(, 1924년 10월 1일~2024년 12월 29일)는 미국의 제39대 대통령 (1977-81)을 지낸 미국의 정치인이다. 민주당 소속으로 1963년부터 1967년까지 조지아주 상원 의원, 1971년부터 1975년까지 조지아주의 76대 주지사을 지냈다. 카터는 100세까지 산 최초의 대통령으로 미국 역사상 가장 장수한 대통령이다.

4번째 줄 : 카터는 조지아주 플레인스에서 태어나고 자랐다. 1946년 미국 해군사관학교를 졸업하고 미국 해군 잠수함에 승선했다. 카터는 군 복무를 마치고 고향으로 돌아와 가족의 땅콩 재배 사업을 되살렸다. 카터는 인종 분리 정책에 반대하며 성장하던 민권 운동을 지지했고, 민주당 내에서 활동가가 되었다. 1963년부터 1967년까지 조지아주 상원 의원으로 재직하였고, 1971년부터 1975년까지 조지아 주지사로 재직했다. 조지아 주 밖에서는 잘 알려지지 않은 다크호스 후보였던 카터는 민주당 후보로 지명되어 1976년 대선에서 공화당의 현직 대통령인 제럴드 포드를 상대로 신승했다.

5번째 줄 : 카터는 취임 둘째 날 베트남 전쟁에서 병역을 기피한 모든 사람들을 사면했다. 에너지부와 교육부를 설립했으며, 에너지 절약, 가격 통제, 신기술을 포함한 국가 에너지 정책을 만들었다. 스태그플레이션에 대응하는 동시에 카터는 캠프 데이비드 협정, 파나마 운하 조약, 제2차 전략 무기 제한 협상을 성공적으로 추진했다. 하지만 임기 말에는 이란 인질 사태, 에너지 위기, 스리마일 섬 사고, 니카라과 혁명, 그리고 소련의 아프가니스탄 침공 등 위기가 이어졌다. 아프가니스탄 침공에 대응하여 카터는 데탕트 정책을 종식시키고 카터 독트린을 선포했으며, 소련에 곡물 금수조치를 부과하고, 1980년 모스크바 하계 올림픽에 대한 다국적 보이콧을 주

In [10]:
# 상위 10 개만 출력
lines[:10]

['<doc id="5" url="https://ko.wikipedia.org/wiki?curid=5" title="지미 카터">',
 '지미 카터',
 '',
 '제임스 얼 "지미 카터 주니어"(, 1924년 10월 1일~2024년 12월 29일)는 미국의 제39대 대통령 (1977-81)을 지낸 미국의 정치인이다. 민주당 소속으로 1963년부터 1967년까지 조지아주 상원 의원, 1971년부터 1975년까지 조지아주의 76대 주지사을 지냈다. 카터는 100세까지 산 최초의 대통령으로 미국 역사상 가장 장수한 대통령이다.',
 '카터는 조지아주 플레인스에서 태어나고 자랐다. 1946년 미국 해군사관학교를 졸업하고 미국 해군 잠수함에 승선했다. 카터는 군 복무를 마치고 고향으로 돌아와 가족의 땅콩 재배 사업을 되살렸다. 카터는 인종 분리 정책에 반대하며 성장하던 민권 운동을 지지했고, 민주당 내에서 활동가가 되었다. 1963년부터 1967년까지 조지아주 상원 의원으로 재직하였고, 1971년부터 1975년까지 조지아 주지사로 재직했다. 조지아 주 밖에서는 잘 알려지지 않은 다크호스 후보였던 카터는 민주당 후보로 지명되어 1976년 대선에서 공화당의 현직 대통령인 제럴드 포드를 상대로 신승했다.',
 '카터는 취임 둘째 날 베트남 전쟁에서 병역을 기피한 모든 사람들을 사면했다. 에너지부와 교육부를 설립했으며, 에너지 절약, 가격 통제, 신기술을 포함한 국가 에너지 정책을 만들었다. 스태그플레이션에 대응하는 동시에 카터는 캠프 데이비드 협정, 파나마 운하 조약, 제2차 전략 무기 제한 협상을 성공적으로 추진했다. 하지만 임기 말에는 이란 인질 사태, 에너지 위기, 스리마일 섬 사고, 니카라과 혁명, 그리고 소련의 아프가니스탄 침공 등 위기가 이어졌다. 아프가니스탄 침공에 대응하여 카터는 데탕트 정책을 종식시키고 카터 독트린을 선포했으며, 소련에 곡물 금수조치를 부과하고, 1980년 모스크바 하계 올림픽에 대한 다국적 보이콧을 주도하여 냉전을 확대했다. 카터는 198

In [12]:
# 몇개의 문장이 존재 하고 , 얼마나 줄었는지
len(result)

38414

In [13]:
# 형태소 분석을 통해 토큰화 진행된 상태이므로 word2vec 을 학습
from gensim.models import Word2Vec
model = Word2Vec(result, vector_size=100, window=5, min_count=5, workers=4, sg=0)

In [14]:
model_result1 = model.wv.most_similar('대한민국')
print(model_result1)

[('한국', 0.7645492553710938), ('연방', 0.7299681901931763), ('미국', 0.7140845060348511), ('일본', 0.7051115036010742), ('법원', 0.7001751661300659), ('캐나다', 0.6951112747192383), ('임시', 0.6948374509811401), ('대구', 0.6912102103233337), ('북한', 0.6902029514312744), ('전국', 0.689179003238678)]


In [18]:
model_result2 = model.wv.most_similar('어벤져스')
print(model_result2)

KeyError: "Key '어벤져스' not present in vocabulary"

어벤져스는 데이터에 없는 단어.

In [17]:
model_result3 = model.wv.most_similar('반도체')
print(model_result3)

[('태블릿', 0.8521904349327087), ('제논', 0.8417340517044067), ('物質', 0.8405523896217346), ('휴대폰', 0.8343572020530701), ('엔지니어링', 0.8288347125053406), ('시스테인', 0.8287860751152039), ('분류학', 0.8280065059661865), ('인터', 0.827658474445343), ('글로벌', 0.8258848190307617), ('ANSI', 0.8246958255767822)]
